# Training a convolutional neural network on the CIFAR-10 dataset using PyTorch.

## Preparing my data
I have downloaded the CIFAR-10 dataset from the University of Toronto's website. I will be using a standard predefined transform to turn the images into tensors, normalizing pixel values to have a mean of 0 and a standard deviation of 1.

In [1]:
from torchvision import transforms

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean = (0.4914, 0.4822, 0.4465), std = (0.2470, 0.2435, 0.2616)) # I used the standard, pre-calculated mean and standard deviation for CIFAR-10
])

I will split the data into three sets: training, validation and testing. I will also apply the transform to the three sets.

In [ ]:
from torch.utils.data import random_split
from torchvision import datasets

training_data, validation_data = random_split(datasets.CIFAR10(root = "./data", train = True, download = False, transform = transform), [45000, 5000])
testing_data = datasets.CIFAR10(root = "./data", train = False, download = False, transform = transform)

print(f"Training data: {len(training_data)}")
print(f"Validation data: {len(validation_data)}")
print(f"Testing data: {len(testing_data)}")
print(f"Classes: {testing_data.classes}")

I will now create the DataLoader object that will feed the data into my model when training. I chose batch_size = 64 because it has seemed like the standard when I have been researching.

In [ ]:
from torch.utils.data import DataLoader

training_loader = DataLoader(training_data, batch_size = 64, shuffle = True)
validation_loader = DataLoader(validation_data, batch_size = 64, shuffle = False)
testing_loader = DataLoader(testing_data, batch_size = 64, shuffle = False)

## Defining my model
I chose 3 convolutional layers because it seemed like a reasonable depth for images this small. My intuition: the first layer picks up on simple things like edges and lines, the second combines those into shapes, and the third combines shapes into more complex patterns.

I doubled the channels at each layer (16 → 32 → 64) based on the same intuition. The first layer doesn't need many features to detect lines, but as you move deeper there are more possible combinations of shapes to represent, so more channels make sense. I also saw this doubling pattern used commonly while researching CNN architectures, which reinforced the choice.

In [ ]:
from torch import nn

class ModelV1(nn.Module):
    def __init__(self):
        super().__init__()

        # Pooling, activation and flattening
        self.pool = nn.MaxPool2d(kernel_size = 2, stride = 2)
        self.relu = nn.ReLU()
        self.flatten = nn.Flatten()

        # Layer 1
        self.conv1 = nn.Conv2d(in_channels = 3, out_channels = 16, kernel_size = 3, padding = 1)
        # ReLU
        # Pooling

        # Layer 2
        self.conv2 = nn.Conv2d(in_channels = 16, out_channels = 32, kernel_size = 3, padding = 1)
        # ReLU
        # Pooling

        # Layer 3
        self.conv3 = nn.Conv2d(in_channels = 32, out_channels = 64, kernel_size = 3, padding = 1)
        # ReLU
        # Pooling

        # Dense layers
        # Flattening
        self.fc1 = nn.Linear(in_features = 4 * 4 * 64, out_features = 64)
        # ReLU
        self.fc2 = nn.Linear(in_features = 64, out_features = 10)
    
    def forward(self, x):
        # Layer 1
        x = self.pool(self.relu(self.conv1(x)))

        # Layer 2
        x = self.pool(self.relu(self.conv2(x)))

        # Layer 3
        x = self.pool(self.relu(self.conv3(x)))

        # Dense layers
        return self.fc2(self.relu(self.fc1(self.flatten(x))))

## Defining my loss function and optimizer
I chose Adam as my optimizer mainly because it was simple to set up and get working. For the learning rate, I used 0.001, which I found to be the commonly recommended default for Adam.

In [ ]:
from torch import optim

model = ModelV1()
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(params = model.parameters(), lr = 0.001)

## Defining my training loop

In [ ]:
from torch import no_grad, max

# History for plotting results
training_losses = []
training_accuracies = []
validation_losses = []
validation_accuracies = []

epochs = 10 # Small number of epochs just to check if the model overfits.

for epoch in range(epochs):
    # Put the model in training mode and reset running loss
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in training_loader:
        # Reset gradients
        optimizer.zero_grad()

        # Forward pass
        predictions = model(images)

        # Calculate loss
        loss = criterion(predictions, labels)

        # Accuracy
        _, prediction = max(predictions, dim = 1)
        correct += (prediction == labels).sum().item()
        total += labels.size(0)

        # Backpropagation
        loss.backward()

        # Adjust weights
        optimizer.step()

        running_loss += loss.item() * images.size(0)

    # Validation
    model.eval()
    validation_loss = 0.0
    validation_correct = 0
    validation_total = 0

    with no_grad():
        for images, labels in validation_loader:
            predictions = model(images)

            loss = criterion(predictions, labels)

            validation_loss += loss.item() * images.size(0)

            _, prediction = max(predictions.data, dim = 1)
            validation_correct += (prediction == labels).sum().item()
            validation_total += labels.size(0)

    # Calculating loss and accuracy for the epoch
    epoch_loss = running_loss / total
    epoch_validation_loss = validation_loss / validation_total
    epoch_accuracy = correct / total
    epoch_validation_accuracy = validation_correct / validation_total

    training_losses.append(epoch_loss)
    training_accuracies.append(epoch_accuracy)
    validation_losses.append(epoch_validation_loss)
    validation_accuracies.append(epoch_validation_accuracy)

    print(f"Epoch [{epoch + 1} / {epochs}]\n"
        f"Training loss: {epoch_loss:.4f}, Training accuracy: {epoch_accuracy * 100:.2f}%\n"
        f"Validation loss: {epoch_validation_loss:.4f}, Validation accuracy: {epoch_validation_accuracy * 100:.2f}%")


## Plotting

In [ ]:
import matplotlib.pyplot as plt

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))

# Graph 1 - Loss
ax1.plot(training_losses, label = 'Training loss', color = 'blue', marker = 'o')
ax1.plot(validation_losses, label = 'Validation loss', color = 'red', marker = 's')
ax1.set_title('Loss by epoch')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.set_xticks(range(epochs))
ax1.set_xticklabels(range(1, epochs + 1))
ax1.legend()
ax1.grid(True)

# Graph 2 - Accuracy
ax2.plot([acc * 100 for acc in training_accuracies], label = 'Training accuracy', color = 'blue', marker='o')
ax2.plot([acc * 100 for acc in validation_accuracies], label = 'Validation accuracy', color='red', marker='s')
ax2.set_title('Accuracy by epoch')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Accuracy')
ax2.set_xticks(range(epochs))
ax2.set_xticklabels(range(1, epochs + 1))
ax2.legend()
ax2.grid(True)

plt.tight_layout()
plt.show()


## Thoughts about the model's performance
The model seems to start overfitting around epoch 3. It seems like it stops learning useful stuff after epoch 6.

## Improving the model
I will start by making two changes:
1. Data augmentation
    I will be applying random flipping and cropping transformations to the images before giving them to my model for training. This makes it less likely that the model will "remember" specific images.
2. Adding dropout
    Dropout layers randomly turn off a percentage of neurons in order to force the network to not rely too much on a single neuron. I will add weak dropout layers between convolutional layers and a stronger one in the dense layers.

In [ ]:
# Data augmentation
from copy import deepcopy

new_transform = transforms.Compose([
    transforms.RandomCrop(size = 32, padding = 2),
    transforms.RandomHorizontalFlip(p = 0.5),
    transforms.ToTensor(),
    transforms.Normalize(mean = (0.4914, 0.4822, 0.4465), std = (0.2470, 0.2435, 0.2616)) # I used the standard, pre-calculated mean and standard deviation for CIFAR-10
])

new_training_data = deepcopy(training_data)
new_training_data.dataset.transform = new_transform

new_training_loader = DataLoader(new_training_data, batch_size = 64, shuffle = True)

In [ ]:
# Redefining the model with dropout
class ModelV2(nn.Module):
    def __init__(self):
        super().__init__()

        # Activation, pooling, dropout and flattening
        self.relu = nn.ReLU()
        self.pool = nn.MaxPool2d(kernel_size = 2, stride = 2)
        self.dropout_light = nn.Dropout(p = 0.25)
        self.dropout_heavy = nn.Dropout(p = 0.5)
        self.flatten = nn.Flatten()

        # Layer 1
        self.conv1 = nn.Conv2d(in_channels = 3, out_channels = 16, kernel_size = 3, padding = 1)
        # ReLU
        # Pooling
        # Light dropout

        # Layer 2
        self.conv2 = nn.Conv2d(in_channels = 16, out_channels = 32, kernel_size = 3, padding = 1)
        # ReLU
        # Pooling
        # Light dropout

        # Layer 3
        self.conv3 = nn.Conv2d(in_channels = 32, out_channels = 64, kernel_size = 3, padding = 1)
        # ReLU
        # Pooling
        # Light dropout

        # Dense layers
        # Flattening
        self.fc1 = nn.Linear(in_features = 4 * 4 * 64, out_features = 64)
        # ReLU
        # Heavy dropout
        self.fc2 = nn.Linear(in_features = 64, out_features = 10)
    
    def forward(self, x):
        # Layer 1
        x = self.dropout_light(self.pool(self.relu(self.conv1(x))))

        # Layer 2
        x = self.dropout_light(self.pool(self.relu(self.conv2(x))))

        # Layer 3
        x = self.dropout_light(self.pool(self.relu(self.conv3(x))))

        # Dense layers
        return self.fc2(self.dropout_heavy(self.relu(self.fc1(self.flatten(x)))))

In [ ]:
model = ModelV2()
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(params = model.parameters(), lr = 0.001)

In [ ]:
# History for plotting results
training_losses = []
training_accuracies = []
validation_losses = []
validation_accuracies = []

epochs = 50 # Runnig for 50 epochs now since regularization slows convergence

for epoch in range(epochs):
    # Put the model in training mode and reset running loss
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in new_training_loader:
        # Reset gradients
        optimizer.zero_grad()

        # Forward pass
        predictions = model(images)

        # Calculate loss
        loss = criterion(predictions, labels)

        # Accuracy
        _, prediction = max(predictions, dim = 1)
        correct += (prediction == labels).sum().item()
        total += labels.size(0)

        # Backpropagation
        loss.backward()

        # Adjust weights
        optimizer.step()

        running_loss += loss.item() * images.size(0)

    # Validation
    model.eval()
    validation_loss = 0.0
    validation_correct = 0
    validation_total = 0

    with no_grad():
        for images, labels in validation_loader:
            predictions = model(images)

            loss = criterion(predictions, labels)

            validation_loss += loss.item() * images.size(0)

            _, prediction = max(predictions.data, dim = 1)
            validation_correct += (prediction == labels).sum().item()
            validation_total += labels.size(0)

    # Calculating loss and accuracy for the epoch
    epoch_loss = running_loss / total
    epoch_validation_loss = validation_loss / validation_total
    epoch_accuracy = correct / total
    epoch_validation_accuracy = validation_correct / validation_total

    training_losses.append(epoch_loss)
    training_accuracies.append(epoch_accuracy)
    validation_losses.append(epoch_validation_loss)
    validation_accuracies.append(epoch_validation_accuracy)

    print(f"Epoch [{epoch + 1} / {epochs}]\n"
        f"Training loss: {epoch_loss:.4f}, Training accuracy: {epoch_accuracy * 100:.2f}%\n"
        f"Validation loss: {epoch_validation_loss:.4f}, Validation accuracy: {epoch_validation_accuracy * 100:.2f}%")


Plotting again.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))

# Graph 1 - Loss
ax1.plot(training_losses, label = 'Training loss', color = 'blue', marker = 'o')
ax1.plot(validation_losses, label = 'Validation loss', color = 'red', marker = 's')
ax1.set_title('Loss by epoch')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.set_xticks(range(epochs))
ax1.set_xticklabels(range(1, epochs + 1))
ax1.legend()
ax1.grid(True)

# Graph 2 - Accuracy
ax2.plot([acc * 100 for acc in training_accuracies], label = 'Training accuracy', color = 'blue', marker='o')
ax2.plot([acc * 100 for acc in validation_accuracies], label = 'Validation accuracy', color='red', marker='s')
ax2.set_title('Accuracy by epoch')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Accuracy')
ax2.set_xticks(range(epochs))
ax2.set_xticklabels(range(1, epochs + 1))
ax2.legend()
ax2.grid(True)

plt.tight_layout()
plt.show()


## Thoughts about the model's performance
The model is no longer overfitting. It seems like it is learning pretty well, but plateaus.

In [ ]:
from torch import save

save({
    'model_state_dict': model.state_dict(),
    'optimizer_state_dict': optimizer.state_dict(),
    'epoch': epoch,
    'val_loss': epoch_validation_loss,
}, "weights.pth")

## Adding CUDA support
I just got back home to my desktop, so I will be able to train on my RTX 5070 instead of my poor ThinkPad's CPU.

## Improving the model
I will be making four changes:
1. Batch normalization
    Batch normalization normalizes a layer's values to have a mean of 0 and a standard deviation of 1. This is supposed to smoothen the loss curve which stabilizes training and allows for higher learning rates.
2. Spatial dropout in convolutional layers
    Spatial dropout randomly turns off a percentage of channels instead of pixels. This is better for convolutional layers since they often contain a lot of pixels that are very similar, removing one doesn't really do much. It is therefore better to leave the pixels alone and turn off entire channels instead.
3. Adding a scheduler to decay my learning rate as the model plateaus.
4. Adding CUDA support
    I just got back home to my desktop, so I will be able to train on my RTX 5070 instead of my poor ThinkPad's CPU.

In [ ]:
# Redefining my model with spatial dropout and batch normalization.
class ModelV3(nn.Module):
    def __init__(self):
        super().__init__()

        # Activation, pooling, dropout and flattening
        self.relu = nn.ReLU()
        self.pool = nn.MaxPool2d(kernel_size = 2, stride = 2)
        self.spatial_dropout_l2 = nn.Dropout2d(p = 0.1)
        self.spatial_dropout_l3 = nn.Dropout2d(p = 0.2)
        self.dropout = nn.Dropout(p = 0.5)
        self.flatten = nn.Flatten()

        # Layer 1
        self.conv1 = nn.Conv2d(in_channels = 3, out_channels = 16, kernel_size = 3, padding = 1)
        self.batchnorm1 = nn.BatchNorm2d(16)
        # ReLU
        # Pooling

        # Layer 2
        self.conv2 = nn.Conv2d(in_channels = 16, out_channels = 32, kernel_size = 3, padding = 1)
        self.batchnorm2 = nn.BatchNorm2d(32)
        # ReLU
        # Pooling
        # Spatial dropout

        # Layer 3
        self.conv3 = nn.Conv2d(in_channels = 32, out_channels = 64, kernel_size = 3, padding = 1)
        self.batchnorm3 = nn.BatchNorm2d(64)
        # ReLU
        # Pooling
        # Spatial dropout

        # Dense layers
        # Flattening
        self.fc1 = nn.Linear(in_features = 4 * 4 * 64, out_features = 64)
        # ReLU
        # Dropout
        self.fc2 = nn.Linear(in_features = 64, out_features = 10)
    
    def forward(self, x):
        # Layer 1
        x = self.pool(self.relu(self.batchnorm1(self.conv1(x))))

        # Layer 2
        x = self.spatial_dropout_l2(self.pool(self.relu(self.batchnorm2(self.conv2(x)))))

        # Layer 3
        x = self.spatial_dropout_l3(self.pool(self.relu(self.batchnorm3(self.conv3(x)))))

        # Dense layers
        return self.fc2(self.dropout(self.relu(self.fc1(self.flatten(x)))))

In [ ]:
# Added scheduler, updated learning rate and number of epochs
# Also added CUDA support
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = ModelV3().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(params = model.parameters(), lr = 0.003)
epochs = 100
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer = optimizer, T_max = epochs)

In [ ]:
# History for plotting results
training_losses = []
training_accuracies = []
validation_losses = []
validation_accuracies = []

for epoch in range(epochs):
    # Put the model in training mode and reset running loss
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in new_training_loader:
        # Send to GPU if I have one
        images, labels = images.to(device), labels.to(device)

        # Reset gradients
        optimizer.zero_grad()

        # Forward pass
        predictions = model(images)

        # Calculate loss
        loss = criterion(predictions, labels)

        # Accuracy
        _, prediction = max(predictions, dim = 1)
        correct += (prediction == labels).sum().item()
        total += labels.size(0)

        # Backpropagation
        loss.backward()

        # Adjust weights
        optimizer.step()

        running_loss += loss.item() * images.size(0)

    # Validation
    model.eval()
    validation_loss = 0.0
    validation_correct = 0
    validation_total = 0

    with no_grad():
        for images, labels in validation_loader:
            # Send to GPU if I have one
            images, labels = images.to(device), labels.to(device)

            predictions = model(images)

            loss = criterion(predictions, labels)

            validation_loss += loss.item() * images.size(0)

            _, prediction = max(predictions.data, dim = 1)
            validation_correct += (prediction == labels).sum().item()
            validation_total += labels.size(0)

    # Calculating loss and accuracy for the epoch
    epoch_loss = running_loss / total
    epoch_validation_loss = validation_loss / validation_total
    epoch_accuracy = correct / total
    epoch_validation_accuracy = validation_correct / validation_total

    training_losses.append(epoch_loss)
    training_accuracies.append(epoch_accuracy)
    validation_losses.append(epoch_validation_loss)
    validation_accuracies.append(epoch_validation_accuracy)

    print(f"Epoch [{epoch + 1} / {epochs}]\n"
        f"Training loss: {epoch_loss:.4f}, Training accuracy: {epoch_accuracy * 100:.2f}%\n"
        f"Validation loss: {epoch_validation_loss:.4f}, Validation accuracy: {epoch_validation_accuracy * 100:.2f}%")

    # Adjust the learning rate
    scheduler.step()

Plotting again, again.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))

# Graph 1 - Loss
ax1.plot(training_losses, label = 'Training loss', color = 'blue', marker = 'o')
ax1.plot(validation_losses, label = 'Validation loss', color = 'red', marker = 's')
ax1.set_title('Loss by epoch')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.set_xticks(range(epochs))
ax1.set_xticklabels(range(1, epochs + 1))
ax1.legend()
ax1.grid(True)

# Graph 2 - Accuracy
ax2.plot([acc * 100 for acc in training_accuracies], label = 'Training accuracy', color = 'blue', marker='o')
ax2.plot([acc * 100 for acc in validation_accuracies], label = 'Validation accuracy', color='red', marker='s')
ax2.set_title('Accuracy by epoch')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Accuracy')
ax2.set_xticks(range(epochs))
ax2.set_xticklabels(range(1, epochs + 1))
ax2.legend()
ax2.grid(True)

plt.tight_layout()
plt.show()
